# Pre-processing — Imputing missing values

**Input:** `kidney_disease.csv` (400 rows × 26 columns)
**Output:** the same frame with one **new `<column>_imputed`** column for every column that
had missing values, so the imputed columns contain **zero nulls**.

**Strategy (fixed):**
* **numeric** columns → **median** (robust to the right-skewed lab values and outliers found in EDA)
* **categorical** columns → **mode** (most frequent category)

Only `pandas` and `numpy` are used.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

## 2. Load the raw data

In [2]:
DATA_PATH = "../kidney_disease.csv"
raw = pd.read_csv(DATA_PATH)
print(f"shape: {raw.shape[0]} rows x {raw.shape[1]} columns")
raw.head()

shape: 400 rows x 26 columns


,id,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,36.0,1.2,NaN,NaN,15.4,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,18.0,0.8,NaN,NaN,11.3,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,53.0,1.8,NaN,NaN,9.6,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,56.0,3.8,111.0,2.5,11.2,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,26.0,1.4,NaN,NaN,11.6,35,7300,4.6,no,no,no,good,no,no,ckd


## 3. Normalise the dirty text values

Before imputing we must turn the hidden defects into real `NaN`s, otherwise they would be treated
as valid categories/numbers:

* strip tabs and surrounding spaces from every text column (`\tno`, ` yes`, `ckd\t`);
* turn the `?` / `\t?` sentinels into `NaN`;
* coerce `pcv`, `wc`, `rc` from text to numeric.

In [3]:
df = raw.copy()

# 1) strip whitespace/tabs and convert sentinels to NaN in all text columns
text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
for c in text_cols:
    df[c] = (df[c].astype("string")
             .str.replace("\t", "", regex=False)
             .str.strip())
    df[c] = df[c].replace({"": pd.NA, "?": pd.NA, "nan": pd.NA})

# 2) these three are numeric but were stored as text
for c in ["pcv", "wc", "rc"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("float64")

print("dtypes after cleaning:")
print(df.dtypes.astype(str).to_string())
print("\ntarget values:", df["classification"].value_counts(dropna=False).to_dict())

dtypes after cleaning:
id                  int64
age               float64
bp                float64
sg                float64
al                float64
su                float64
rbc                string
pc                 string
pcc                string
ba                 string
bgr               float64
bu                float64
sc                float64
sod               float64
pot               float64
hemo              float64
pcv               float64
wc                float64
rc                float64
htn                string
dm                 string
cad                string
appet              string
pe                 string
ane                string
classification     string

target values: {'ckd': 250, 'notckd': 150}


In [4]:
before = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "nulls": df.isna().sum(),
    "null_%": (df.isna().mean() * 100).round(2),
})
before[before["nulls"] > 0].sort_values("nulls", ascending=False)

,dtype,nulls,null_%
rbc,string,152,38.00
rc,float64,131,32.75
wc,float64,106,26.50
pot,float64,88,22.00
sod,float64,87,21.75
pcv,float64,71,17.75
pc,string,65,16.25
hemo,float64,52,13.00
su,float64,49,12.25
sg,float64,47,11.75


## 4. Impute: median for numeric, mode for categorical

For every column that still has missing values we add a **new column** named
`<column>_imputed`. The original column is left untouched so the filled values can be audited.

In [5]:
# detect types dynamically from the cleaned frame
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in df.columns if c not in numeric_cols]

# only columns that actually contain missing values
null_cols = [c for c in df.columns if df[c].isna().any()]

print("numeric columns     :", numeric_cols)
print("categorical columns :", categorical_cols)
print("columns to impute   :", null_cols)

numeric columns     : ['id', 'age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc']
categorical columns : ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification']
columns to impute   : ['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']


In [6]:
log_rows = []

for c in null_cols:
    n_before = int(df[c].isna().sum())

    if c in numeric_cols:
        strategy = "median"
        fill_value = df[c].median()
    else:
        strategy = "mode"
        modes = df[c].mode(dropna=True)
        fill_value = modes.iloc[0] if len(modes) else pd.NA

    new_col = f"{c}_imputed"
    df[new_col] = df[c].fillna(fill_value)

    log_rows.append({
        "column": c,
        "type": "numeric" if c in numeric_cols else "categorical",
        "nulls_before": n_before,
        "null_%": round(n_before / len(df) * 100, 2),
        "strategy": strategy,
        "fill_value": fill_value,
        "new_column": new_col,
        "nulls_after": int(df[new_col].isna().sum()),
    })

imputation_log = pd.DataFrame(log_rows)
imputation_log

,column,type,nulls_before,null_%,strategy,fill_value,new_column,nulls_after
0,age,numeric,9,2.25,median,55.0,age_imputed,0
1,bp,numeric,12,3.00,median,80.0,bp_imputed,0
2,sg,numeric,47,11.75,median,1.02,sg_imputed,0
3,al,numeric,46,11.50,median,0.0,al_imputed,0
4,su,numeric,49,12.25,median,0.0,su_imputed,0
5,rbc,categorical,152,38.00,mode,normal,rbc_imputed,0
6,pc,categorical,65,16.25,mode,normal,pc_imputed,0
7,pcc,categorical,4,1.00,mode,notpresent,pcc_imputed,0
8,ba,categorical,4,1.00,mode,notpresent,ba_imputed,0
9,bgr,numeric,44,11.00,median,121.0,bgr_imputed,0


**Reading the log:** `fill_value` is the median (numeric) or the most frequent category (categorical)
that was used; `nulls_after` must be `0` for every new column.

In [7]:
imputed_cols = [f"{c}_imputed" for c in null_cols]

print("nulls still present in the new *_imputed columns:",
      int(df[imputed_cols].isna().sum().sum()))
print("nulls still present in the original columns      :",
      int(df[null_cols].isna().sum().sum()))

df[imputed_cols].isna().sum().to_frame("nulls_after_imputation")

nulls still present in the new *_imputed columns: 0
nulls still present in the original columns      : 1012


,nulls_after_imputation
age_imputed,0
bp_imputed,0
sg_imputed,0
al_imputed,0
su_imputed,0
rbc_imputed,0
pc_imputed,0
pcc_imputed,0
ba_imputed,0
bgr_imputed,0


## 5. Build the modelling-ready frame

Drop the original (still-null) columns and keep their imputed versions under the original names,
so the result has the same 26-column schema with **no missing values at all**.

In [8]:
model_df = df.drop(columns=null_cols).copy()
for c in null_cols:
    model_df[c] = df[f"{c}_imputed"]

# restore the original column order
model_df = model_df[raw.columns]

print(f"model_df shape: {model_df.shape}")
print("total missing values in model_df:", int(model_df.isna().sum().sum()))
print("duplicate columns:", int(model_df.columns.duplicated().sum()))
model_df.head()

model_df shape: (400, 26)
total missing values in model_df: 0
duplicate columns: 0


,id,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,normal,normal,notpresent,notpresent,121.0,36.0,1.2,138.0,4.4,15.4,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,normal,normal,notpresent,notpresent,121.0,18.0,0.8,138.0,4.4,11.3,38.0,6000.0,4.8,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,53.0,1.8,138.0,4.4,9.6,31.0,7500.0,4.8,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,56.0,3.8,111.0,2.5,11.2,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,26.0,1.4,138.0,4.4,11.6,35.0,7300.0,4.6,no,no,no,good,no,no,ckd


In [9]:
final_check = pd.DataFrame({
    "dtype": model_df.dtypes.astype(str),
    "nulls": model_df.isna().sum(),
    "null_%": (model_df.isna().mean() * 100).round(2),
})
final_check

,dtype,nulls,null_%
id,int64,0,0.0
age,float64,0,0.0
bp,float64,0,0.0
sg,float64,0,0.0
al,float64,0,0.0
su,float64,0,0.0
rbc,string,0,0.0
pc,string,0,0.0
pcc,string,0,0.0
ba,string,0,0.0


## 6. What changed / next steps

* **New columns:** one `*_imputed` column per column that had nulls — all with zero nulls.
* **Original columns preserved:** `df` still holds the raw (unfilled) values next to the imputed ones for audit.
* **`model_df`:** fully imputed, original 26-column schema, zero missing values — ready for encoding/scaling/splitting.

Notes to keep in mind for modelling:

* `rbc` was ~38% missing and its missingness is informative (mostly CKD rows), so a
  `rbc_was_missing` flag may still be worth adding before dropping the original column.
* `pcv`, `wc`, `rc` are now numeric; `classification` is still the text label (`ckd` / `notckd`)
  and should be encoded (e.g. `ckd → 1`, `notckd → 0`) before training.
* Median/mode fills are computed on the **whole dataset** here for exploration; for a leakage-free
  model, fit the fill values on the **training split only** and apply them to validation/test.